In [2]:
import heapq

# Hướng di chuyển
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

# Chuyển ma trận -> tuple (để lưu visited)
def mat_to_tuple(mat):
    return tuple(num for row in mat for num in row)

# Heuristic: Manhattan Distance (tổng quát)
def calculate_cost(mat, goal_pos, n):
    cost = 0
    for i in range(n):
        for j in range(n):
            value = mat[i][j]
            if value != 0:
                x, y = goal_pos[value]
                cost += abs(x - i) + abs(y - j)
    return cost

class Node:
    def __init__(self, parent, mat, empty_pos, level, cost):
        self.parent = parent
        self.mat = mat
        self.empty_pos = empty_pos
        self.level = level  # g(n)
        self.cost = cost    # h(n)

    def __lt__(self, other):
        return (self.cost + self.level) < (other.cost + other.level)

def new_node(mat, empty_pos, new_pos, level, parent, goal_pos, n):
    new_mat = [row[:] for row in mat]

    x1, y1 = empty_pos
    x2, y2 = new_pos

    # hoán đổi
    new_mat[x1][y1], new_mat[x2][y2] = new_mat[x2][y2], new_mat[x1][y1]

    cost = calculate_cost(new_mat, goal_pos, n)
    return Node(parent, new_mat, new_pos, level, cost)

def is_safe(x, y, n):
    return 0 <= x < n and 0 <= y < n

def print_matrix(mat):
    for row in mat:
        print(*row)
    print()

def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.mat)

def solve(initial, goal, empty_pos, n):
    pq = []
    visited = set()

    # tạo bảng vị trí đích để tăng tốc heuristic
    goal_pos = {}
    for i in range(n):
        for j in range(n):
            goal_pos[goal[i][j]] = (i, j)

    cost = calculate_cost(initial, goal_pos, n)
    root = Node(None, initial, empty_pos, 0, cost)

    heapq.heappush(pq, root)

    while pq:
        current = heapq.heappop(pq)

        if current.cost == 0:
            print("=== TÌM THẤY LỜI GIẢI ===")
            print_path(current)
            print("Số bước:", current.level)
            return

        visited.add(mat_to_tuple(current.mat))

        for i in range(4):
            new_x = current.empty_pos[0] + rows[i]
            new_y = current.empty_pos[1] + cols[i]

            if is_safe(new_x, new_y, n):
                new_mat = [row[:] for row in current.mat]
                new_mat[current.empty_pos[0]][current.empty_pos[1]], new_mat[new_x][new_y] = \
                    new_mat[new_x][new_y], new_mat[current.empty_pos[0]][current.empty_pos[1]]

                if mat_to_tuple(new_mat) not in visited:
                    child = new_node(
                        current.mat,
                        current.empty_pos,
                        [new_x, new_y],
                        current.level + 1,
                        current,
                        goal_pos,
                        n
                    )
                    heapq.heappush(pq, child)

    print("Không tìm thấy lời giải!")

if __name__ == "__main__":
    n = 5

    initial = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9,10],
        [11,12,13,14,15],
        [16,17,18, 0,19],
        [21,22,23,24,20]
    ]

    goal = [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9,10],
        [11,12,13,14,15],
        [16,17,18,19,20],
        [21,22,23,24, 0]
    ]

    empty_pos = [3, 3]

    solve(initial, goal, empty_pos, n)

=== TÌM THẤY LỜI GIẢI ===
1 2 3 4 5
6 7 8 9 10
11 12 13 14 15
16 17 18 0 19
21 22 23 24 20

1 2 3 4 5
6 7 8 9 10
11 12 13 14 15
16 17 18 19 0
21 22 23 24 20

1 2 3 4 5
6 7 8 9 10
11 12 13 14 15
16 17 18 19 20
21 22 23 24 0

Số bước: 2
